# Overview - Data Preparation

This first stage loosely follows the CRISP-DM methodology for data mining. Business understanding is limited to the information provided in the `Reach for Change` documentation provided in [Descriptive/Handout_Guidelines_DSML.pdf](Handout_Guidelines_DSML.pdf).
The rest of this section on:

- Data understanding: An exploratory analysis of the dataset
- Data preparation: Preparation of the dataset for the descriptive and predictive analysis.
  -  Drop unique identifiers
  - Impute missing demographic values
  - Address highly correlated financial features.


## Step 1: Establish the project root
The reviewers use Colab, a good number of professionals use local IDEs. The next cell estabilishes support for the data directory for both environments.

In [137]:
from pathlib import Path

import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    lab_root = Path("/content/drive/MyDrive/Colab Notebooks/DSML/LAB04 - Building a Predictive Model Part II")
else:
    lab_root = Path(".")

print(f"lab_root set to: {lab_root.resolve()}")

lab_root set to: /Users/keni/dev/nova/machine-learning/predictive-project


## Step 2: Data Exploration

The following cells explore the data and note the necessary interventions for the preparatory stage.

### 2.1: Dataset-level exploration
In the first stage, the whole dataset is explored to understand it's dimensions, the integrity of data and cleanliness:
- Shape
- Duplicate and missing values
- Data type issues, if any

### 2.2: Column-level exploration
This stage the categorical and numerical columns will be treated separately. Categorical will be analyzed first because the presence of bad data may cause numerical columns to be classified as categorical.

- Categorical columns:
  - Cardinality check
  - Check for class imbalance
  - Consider the encoding options for nominal and ordinal values
- Numerical columns:
  - Review missing values per column: consider the percentages and if it will be dropped or inferred.
  - Consider the distribution: plot histograms and calculate skewness since our models will perform poorly on skewed data. If some columns are skewed, transformations will be needed.
  - Check for Scale and magnitudes since distance-based models like K-Means are susceptible to these. The data will be normalized as required.
  - Detect outliers

### 2.3: Explore the relationships and interactions in the columns
- Collinearity analysis with Spearman/Pearson
- Cluster assessments: *Clustering
- Feature/target relationships: *Prediction


In [138]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [139]:
data_path = lab_root / "data" / "donors_descriptive.csv"
if not data_path.exists():
    raise FileNotFoundError(
        f"Input file not found. Expected: {data_path}."
    )

df = pd.read_csv(data_path)
df.head()

,CONTROL_NUMBER,CARD_PROM_12,CHILDREN,DONOR_AGE,DONOR_GENDER,FILE_CARD_GIFT,FREQUENCY_STATUS_97NK,HOME_OWNER,INCOME_GROUP,LAST_GIFT_AMT,...,RECENT_AVG_CARD_GIFT_AMT,RECENT_AVG_GIFT_AMT,RECENT_CARD_RESPONSE_COUNT,RECENT_CARD_RESPONSE_PROP,RECENT_RESPONSE_COUNT,RECENT_RESPONSE_PROP,RECENT_STAR_STATUS,SES,URBANICITY,WEALTH_RATING
0,61745,4.0,2.000000,33.0,M,0.0,1.0,H,5.0,20.0,...,0.00,17.50,0.0,0.000,2.0,0.154,0.0,2,T,NaN
1,112703,3.0,-2.341107,NaN,F,1.0,1.0,U,NaN,15.0,...,15.00,15.00,1.0,0.250,1.0,0.100,0.0,3,R,NaN
2,166437,4.0,0.000000,74.0,F,7.0,3.0,H,4.0,10.0,...,0.00,10.67,0.0,0.000,3.0,0.231,1.0,1,U,NaN
3,170621,4.0,4.000000,61.0,M,13.0,1.0,H,6.0,11.0,...,10.00,10.00,2.0,0.286,2.0,0.111,0.0,1,U,NaN
4,44428,6.0,3.000000,75.0,M,3.0,4.0,H,3.0,7.0,...,5.67,5.40,3.0,0.600,5.0,0.500,0.0,2,C,NaN


## 2.1: Dataset Level Explorations

In [140]:
print(f"dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"duplicate rows: {df.duplicated().sum()}")

dataset shape: 13560 rows x 40 columns
duplicate rows: 0


In [141]:
print("--- column names and data types ---")
print(df.info())

--- column names and data types ---
<class 'pandas.DataFrame'>
RangeIndex: 13560 entries, 0 to 13559
Data columns (total 40 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CONTROL_NUMBER               13560 non-null  int64  
 1   CARD_PROM_12                 13297 non-null  float64
 2   CHILDREN                     13289 non-null  float64
 3   DONOR_AGE                    9970 non-null   float64
 4   DONOR_GENDER                 13303 non-null  str    
 5   FILE_CARD_GIFT               13292 non-null  float64
 6   FREQUENCY_STATUS_97NK        13296 non-null  float64
 7   HOME_OWNER                   13292 non-null  str    
 8   INCOME_GROUP                 10273 non-null  float64
 9   LAST_GIFT_AMT                13296 non-null  float64
 10  LIFETIME_CARD_PROM           13289 non-null  float64
 11  LIFETIME_GIFT_AMOUNT         13288 non-null  float64
 12  LIFETIME_GIFT_COUNT          13304 non-null  floa

In [142]:
print("--- columns without missing data --- ")
print(df.columns[df.isnull().any()==False])
print("--- unique columns --- ")
print(df.columns[df.nunique() == len(df)].tolist())

--- columns without missing data --- 
Index(['CONTROL_NUMBER'], dtype='str')
--- unique columns --- 
['CONTROL_NUMBER']


In [143]:
print("--- columns where % missing values > 0 ---\n")
missing_pct = df.isnull().mean() * 100
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

--- columns where % missing values > 0 ---

WEALTH_RATING                  46.231563
DONOR_AGE                      26.474926
INCOME_GROUP                   24.240413
MONTHS_SINCE_LAST_PROM_RESP     3.171091
SES                             2.109145
PCT_ATTRIBUTE3                  2.079646
URBANICITY                      2.050147
RECENT_CARD_RESPONSE_COUNT      2.050147
MONTHS_SINCE_FIRST_GIFT         2.050147
PCT_ATTRIBUTE1                  2.035398
LIFETIME_MAX_GIFT_AMT           2.035398
RECENT_STAR_STATUS              2.028024
MONTHS_SINCE_LAST_GIFT          2.028024
RECENT_CARD_RESPONSE_PROP       2.020649
LIFETIME_PROM                   2.020649
NUMBER_PROM_12                  2.005900
LIFETIME_GIFT_AMOUNT            2.005900
LIFETIME_CARD_PROM              1.998525
CHILDREN                        1.998525
RECENT_AVG_GIFT_AMT             1.976401
FILE_CARD_GIFT                  1.976401
HOME_OWNER                      1.976401
PCT_ATTRIBUTE4                  1.969027
RECENT_RESPON

## 2.1 Insights

- Small dataset: 13560 rows and 40 columns
- No duplicate records
- Column `CONTROL_NUMBER` is unique and has no missing records. It's sparse and will not be useful for clustering or prediction but we'll make it the index of the dataframe.
- 3 colums have the highest percentage of missing data: 

| Column        | Percentage Missing |
|---------------|--------------------|
| WEALTH_RATING | 46.23              |
| DONOR AGE     | 26.47              |
| INCOME GROUP  | 24.24              |

## 2.2 Categorical Columns Analysis
- Categorical columns:
  - Explore miscategorized column data types i.e. numerical misclassified as categorical because of the presence of bad data or nature of data. 
  - Cardinality check
  - Check for class imbalance: rare classes may need to be grouped together (*clustering), sever inbalance may need weighting (*prediction)
  - Consider the encoding options for:
    - nominal (no inherent classification e.g. city names, eye colors)
    - ordinal (clear & meaning order or hierarchy e.g. income class)

In [144]:
# Explore categorical columns

categorical_cols = df.select_dtypes(include=['object', 'str', 'category']).columns.tolist()
print("--- categorical columns to audit ---")
print(categorical_cols)

--- categorical columns to audit ---
['DONOR_GENDER', 'HOME_OWNER', 'RECENCY_STATUS_96NK', 'SES', 'URBANICITY']


In [145]:
print("--- donor options ---")
print(df["DONOR_GENDER"].value_counts(dropna=False))

print("--- home owner options ---")
print(df["HOME_OWNER"].value_counts(dropna=False))

print("--- recency status 96nk options ---")
print(df["RECENCY_STATUS_96NK"].value_counts(dropna=False))

print("--- ses options ---")
print(df["SES"].value_counts(dropna=False))

print("--- urbanicity options ---")
print(df["URBANICITY"].value_counts(dropna=False))

--- donor options ---
DONOR_GENDER
F      7222
M      5376
U       705
NaN     257
Name: count, dtype: int64
--- home owner options ---
HOME_OWNER
H      7251
U      6041
NaN     268
Name: count, dtype: int64
--- recency status 96nk options ---
RECENCY_STATUS_96NK
A      8202
S      2877
F      1048
N       825
E       283
NaN     266
L        59
Name: count, dtype: int64
--- ses options ---
SES
2      6365
1      4091
3      2277
?       292
NaN     286
4       249
Name: count, dtype: int64
--- urbanicity options ---
URBANICITY
S      3103
C      2741
T      2734
R      2732
U      1677
?       295
NaN     278
Name: count, dtype: int64


Note:
- SES contains categorical ordinal values
  - Missing values need to be handled (`?`). It constitute 2.1% of our records so we won't delete but infer from other columns including HOME_OWNER, INCOME_GROUP, WEALTH_RATING, MEDIAN_HOME_VALUE, MEDIAN_HOUSEHOLD_INCOME.
- DONOR_GENDER (nominal values): 
  - 5.3% of the records NaN
  - Contains `M`, `F` and `U`. We'll treat `U` + `NAN` as a different category  (could be privacy conscious individuals etc)
- HOME_OWNER 
  - This will be treated as a nominal value because the `U` is unknown, so the `H` is not `higher`.
- RECENCY_STATUS_96NK
  - It's ordinal, and has a clear progression from most active to least active.
- URBANICITY
  - It's ordinal, has a clear progression of population density
  - Missing about 2% of values, we will infer this from similar columns. MEDIAN_HOME_VALUE is a good candidate.


## 2.2 Numerical columns Analysis
  - Explore miscategorized column data types i.e. norminal and ordinal attributes misclassified as numerical. They may need further processing like scaling.
  - Review missing values per column: consider the percentages and if it will be dropped or inferred.
  - Consider the distribution: plot histograms and calculate skewness since our models will perform poorly on skewed data. If some columns are skewed, transformations will be needed.
  - Check for Scale and magnitudes since distance-based models like K-Means are susceptible to these. The data will be normalized as required.
  - Detect outliers

In [146]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print("--- numerical columns to audit ---")
print(numerical_cols)

--- numerical columns to audit ---
['CONTROL_NUMBER', 'CARD_PROM_12', 'CHILDREN', 'DONOR_AGE', 'FILE_CARD_GIFT', 'FREQUENCY_STATUS_97NK', 'INCOME_GROUP', 'LAST_GIFT_AMT', 'LIFETIME_CARD_PROM', 'LIFETIME_GIFT_AMOUNT', 'LIFETIME_GIFT_COUNT', 'LIFETIME_MAX_GIFT_AMT', 'LIFETIME_MIN_GIFT_AMT', 'LIFETIME_PROM', 'MEDIAN_HOME_VALUE', 'MEDIAN_HOUSEHOLD_INCOME', 'MONTHS_SINCE_FIRST_GIFT', 'MONTHS_SINCE_LAST_GIFT', 'MONTHS_SINCE_LAST_PROM_RESP', 'NUMBER_PROM_12', 'PCT_ATTRIBUTE1', 'PCT_ATTRIBUTE2', 'PCT_ATTRIBUTE3', 'PCT_ATTRIBUTE4', 'PCT_OWNER_OCCUPIED', 'PEP_STAR', 'PER_CAPITA_INCOME', 'RECENT_AVG_CARD_GIFT_AMT', 'RECENT_AVG_GIFT_AMT', 'RECENT_CARD_RESPONSE_COUNT', 'RECENT_CARD_RESPONSE_PROP', 'RECENT_RESPONSE_COUNT', 'RECENT_RESPONSE_PROP', 'RECENT_STAR_STATUS', 'WEALTH_RATING']


In [147]:
# find low cardinality numerical columns, inspect if they are actually categorical.
num_unique = df[numerical_cols].nunique().sort_values()
print("--- numerical columns and unique count ---")
print(num_unique)


--- numerical columns and unique count ---
PEP_STAR                           4
FREQUENCY_STATUS_97NK              6
CHILDREN                           7
INCOME_GROUP                       9
WEALTH_RATING                     12
RECENT_CARD_RESPONSE_COUNT        12
CARD_PROM_12                      18
RECENT_RESPONSE_COUNT             18
RECENT_STAR_STATUS                25
MONTHS_SINCE_LAST_GIFT            26
FILE_CARD_GIFT                    33
MONTHS_SINCE_LAST_PROM_RESP       36
RECENT_CARD_RESPONSE_PROP         41
NUMBER_PROM_12                    52
LIFETIME_CARD_PROM                53
PCT_ATTRIBUTE1                    70
LIFETIME_GIFT_COUNT               73
LIFETIME_MIN_GIFT_AMT             75
DONOR_AGE                         82
PCT_ATTRIBUTE2                    82
PCT_ATTRIBUTE3                    88
LAST_GIFT_AMT                     93
PCT_ATTRIBUTE4                   101
PCT_OWNER_OCCUPIED               102
RECENT_RESPONSE_PROP             113
LIFETIME_MAX_GIFT_AMT           

In [148]:
print("--- PEP_STAR options ---")
print(df["PEP_STAR"].value_counts(dropna=False))

--- PEP_STAR options ---
PEP_STAR
 1.000000    6615
 0.000000    6547
 NaN          265
-0.529861      70
 2.003609      63
Name: count, dtype: int64


In [149]:
print("--- FREQUENCY_STATUS_97NK ---")
print(df["FREQUENCY_STATUS_97NK"].value_counts(dropna=False))

--- FREQUENCY_STATUS_97NK ---
FREQUENCY_STATUS_97NK
 1.000000    6148
 2.000000    2851
 3.000000    2274
 4.000000    1889
 NaN          264
-1.365242      70
 5.283869      64
Name: count, dtype: int64


In [150]:
print("--- CHILDREN ---")
print(df["CHILDREN"].value_counts(dropna=False))

--- CHILDREN ---
CHILDREN
 1.000000    2658
 4.000000    2646
 2.000000    2631
 3.000000    2630
 0.000000    2581
 NaN          271
-2.341107      75
 6.264709      68
Name: count, dtype: int64


In [151]:
print("--- INCOME_GROUP ---")
print(df["INCOME_GROUP"].value_counts(dropna=False))

--- INCOME_GROUP ---
INCOME_GROUP
 NaN         3287
 5.000000    2177
 2.000000    1758
 4.000000    1732
 1.000000    1217
 3.000000    1133
 6.000000    1076
 7.000000    1050
 9.490799      69
-1.728999      61
Name: count, dtype: int64


In [152]:
print("--- WEALTH_RATING ---")
print(df["WEALTH_RATING"].value_counts(dropna=False))

--- WEALTH_RATING ---
WEALTH_RATING
 NaN          6269
 9.000000      931
 8.000000      840
 7.000000      820
 5.000000      745
 6.000000      736
 3.000000      722
 4.000000      712
 2.000000      643
 1.000000      560
 0.000000      445
 13.446890      70
-3.751835       67
Name: count, dtype: int64


In [153]:
print("--- RECENT_CARD_RESPONSE_COUNT ---")
print(df["RECENT_CARD_RESPONSE_COUNT"].value_counts(dropna=False))


--- RECENT_CARD_RESPONSE_COUNT ---
RECENT_CARD_RESPONSE_COUNT
 1.000000    4382
 2.000000    2867
 0.000000    2641
 3.000000    1610
 4.000000     823
 5.000000     432
 NaN          278
 6.000000     219
 7.000000     114
-3.703178      73
 6.340076      70
 8.000000      38
 9.000000      13
Name: count, dtype: int64


In [154]:
print("--- CARD_PROM_12 ---")
print(df["CARD_PROM_12"].value_counts(dropna=False))

--- CARD_PROM_12 ---
CARD_PROM_12
6.000000     7025
5.000000     2475
4.000000     2306
3.000000      499
NaN           263
7.000000      230
2.000000      225
8.000000      138
11.000000      88
9.155742       70
2.136786       66
9.000000       65
10.000000      42
1.000000       41
12.000000      16
13.000000       7
17.000000       2
14.000000       1
15.000000       1
Name: count, dtype: int64


In [155]:
print("--- RECENT_RESPONSE_COUNT ---")
print(df["RECENT_RESPONSE_COUNT"].value_counts(dropna=False))

--- RECENT_RESPONSE_COUNT ---
RECENT_RESPONSE_COUNT
 2.000000     3388
 1.000000     2966
 3.000000     2334
 4.000000     1696
 5.000000     1070
 6.000000      652
 7.000000      383
 NaN           266
 8.000000      238
 0.000000      164
 9.000000      111
 10.000000      76
 9.145253       68
-3.222148       66
 11.000000      40
 12.000000      22
 13.000000      17
 14.000000       2
 16.000000       1
Name: count, dtype: int64


In [156]:
print("--- RECENT_STAR_STATUS ---")
print(df["RECENT_STAR_STATUS"].value_counts(dropna=False))

--- RECENT_STAR_STATUS ---
RECENT_STAR_STATUS
 0.000000     9002
 1.000000     2913
 NaN           275
 3.000000      242
 4.000000      222
 5.000000      138
 11.000000     120
 7.000000       99
 12.000000      76
-7.806416       73
 8.589483       63
 6.000000       59
 10.000000      56
 13.000000      49
 14.000000      41
 8.000000       27
 19.000000      21
 15.000000      18
 2.000000       18
 16.000000      11
 17.000000      11
 18.000000       8
 21.000000       7
 9.000000        6
 20.000000       4
 22.000000       1
Name: count, dtype: int64


Notes:

- The description of the other columns suggest the are discrete numerical or purely numerical
- DONOR_AGE has 82 distinct values, it needs additional investigation.

In [157]:
print("--- DONOR_AGE ---")
print(df["DONOR_AGE"].value_counts(dropna=False).sort_index())

print("--- DONOR_AGES >= 90 and <= 18")
print(((df['DONOR_AGE'] >= 90) | (df['DONOR_AGE'] <= 18)).sum())

print("--- DONOR_AGES NAN")
missing_ages = df['DONOR_AGE'].isna().sum()
print(missing_ages)


--- DONOR_AGE ---
DONOR_AGE
0.000000         2
2.000000         3
4.000000         1
6.000000         5
7.000000        49
              ... 
85.000000      114
86.000000       92
87.000000      110
108.934261      73
NaN           3590
Name: count, Length: 83, dtype: int64
--- DONOR_AGES >= 90 and <= 18
271
--- DONOR_AGES NAN
3590


Notes:
- CONTROL_NUMBER: Already identified as sparse, completely unique and to be used as our index.
- PEP_STAR
  - categorical with only 4 options. However PEP_STAR values should be a binary 0 or 1 by definition so we will fix the < 1% issues (-0.5 and 2). We will make the - values 0 and the positive to 1.
- FREQUENCY_STATUS_97NK
  - ordinal, values 1-5. requires cleaning. We'll assume -1 should be 1
  - there are very few `5`, less that one and not distributed like the values 1-4, we'll bucket into 1-4
  - NAN values will be inferred from INCOME_GROUP, the description suggests it's useful to determine no. of times donated in the past.
- CHILDREN: 
  - we'll assume these are ordinal values, given people with more children may be less able to donate.
  - we'll fix the bad data and round to integer values
  - we do not have any 5, and have a few values at 6. we will mvoe the 6 into the 4 and cap the data at 0-4
- INCOME_GROUP
  - ordinal
  - clear values from 1 - 7. A few anomalies of -1.7xx and 9.xx making less that 1% of data
  - we will clip these into the 1-7 range and store the -ve values as NaN for now.
  - 24% missing values
- WEALTH_RATING
  - ordinal
  - 46% missing
  - most values between 0-9, <1% showing -3.xxx and 13.xxx will be clipped into the 0-9 buckets.
  - at this stage, we'll put -1 for the anomalies and handle at the model layer, depending on the model.
- RECENT_CARD_RESPONSE_COUNT
  - discrete numerical, dont't scale
  - fix negative values by making them zero.
- CARD_PROM_12: discrete numerical, round to integer, scale*
- RECENT_RESPONSE_COUNT: 
  - discrete numerical. scale*
  - has some negative values, <1%, we clip to zero.
- RECENT_STAR_STATUS: 
  - if star status was achieved in the last 4 years, meaning binary 0 or 1 expected?
  - has 25 numerical values instead. 88% are 0 or 1 binary values.
  - we'll make the negative values 0 and >0 to be 1
- DONOR_AGE
  - continuous numerical column
  - Lots of missing ages, 26%. Approx. 2% < 18 and >90
  - We'll clip the higher range to 90, assuming the behaviour of our older (oldest) donors are similar
  - We'll treat values less than 18 as missing values. Then infer (below)
  - We decide to infer age with other income metrics (INCOME_GROUP, WEALTH_RATING, MEDIAN_HOME_VALUE)
  - We'll use KNN Inputer!
- The remaining columns:
  - Columns: 
    discreet columns
    - MONTHS_SINCE_LAST_GIFT
    - FILE_CARD_GIFT
    - MONTHS_SINCE_LAST_PROM_RESP
    - MONTHS_SINCE_FIRST_GIFT
    - LIFETIME_PROM
    - NUMBER_PROM_12
    - LIFETIME_CARD_PROM
    - LIFETIME_GIFT_COUNT

    %tage based
    - RECENT_CARD_RESPONSE_PROP
    - RECENT_RESPONSE_PROP
    - PCT_ATTRIBUTE1
    - PCT_ATTRIBUTE2
    - PCT_ATTRIBUTE3
    - PCT_ATTRIBUTE4
    - PCT_OWNER_OCCUPIED

    monetary columns
    - LIFETIME_MIN_GIFT_AMT
    - LAST_GIFT_AMT
    - LIFETIME_MAX_GIFT_AMT
    - RECENT_AVG_CARD_GIFT_AMT
    - RECENT_AVG_GIFT_AMT
    - LIFETIME_GIFT_AMOUNT
    - MEDIAN_HOUSEHOLD_INCOME
    - MEDIAN_HOME_VALUE
    - PER_CAPITA_INCOME
  - Treatment
    - discrete: 
      - whole numbers, nearest whole number clipping
      - cannot be negative, clip to 0
    - percentage-based
      - between 0.0 and 1.0 or 0 - 100 %
      - clip values between these boundaries
    - monetary
      - cannot be negative
      - >= 0 or actually > 0


## Step 3: Data Cleaning

We will apply all the observations above to the dataset

In [158]:
print("--- DONOR_GENDER -- Start --- ")
df['DONOR_GENDER'] = df['DONOR_GENDER'].fillna('U').str.upper()
print("--- DONOR_GENDER -- End --- ")

print("--- HOME_OWNER -- Start ---")
df['HOME_OWNER'] = df['HOME_OWNER'].fillna('U').str.upper()
df['HOME_OWNER'] = df['HOME_OWNER'].map({'H': 1, 'U': 0}).astype(int)
print("--- HOME_OWNER -- End ---")

print("--- RECENCY_STATUS_96NK -- Start ---")
df['RECENCY_STATUS_96NK'] = df['RECENCY_STATUS_96NK'].fillna('U').str.upper()
print("--- RECENCY_STATUS_96NK -- End ---")

print("--- SES -- Start ---")
df['SES'] = df['SES'].replace('?', np.nan).astype(float)
df['INC_BUCKET'] = pd.qcut(df['MEDIAN_HOUSEHOLD_INCOME'], q=4, labels=False, duplicates='drop')
# 3. Create a clean subset of rows that contain real, un-imputed SES data
clean_subset = df[df['SES'].notna()]
verified_ses_medians = clean_subset.groupby('INC_BUCKET')['SES'].median()
df['SES'] = df['SES'].fillna(df['INC_BUCKET'].map(verified_ses_medians))
df['SES'] = df['SES'].fillna(df['SES'].median()).round().astype(int) # Fallback
df.drop(columns=['INC_BUCKET'], inplace=True)
print("--- SES -- End ---")

print("--- URBANICITY -- Start ---")
df['URBANICITY'] = df['URBANICITY'].replace('?', None).fillna('Unknown').str.upper()
print("--- URBANICITY -- End ---")

# numerical?
print("--- PEP_STAR ---")
df['PEP_STAR'] = df['PEP_STAR'].apply(lambda x: np.nan if x < 0 else (1.0 if x > 1 else x))
df['PEP_STAR_IS_MISSING'] = np.where(df['PEP_STAR'].isna(), 1, 0) # preserve missingness as a feature, since it may be informative
df['PEP_STAR'] = np.where(df['PEP_STAR'] > 0.5, 1, 0).astype(int)
# 2. Bucket lifetime gift counts into 3 tiers (Low, Medium, High frequency)
df['GIFT_COUNT_TIER'] = pd.qcut(df['LIFETIME_GIFT_COUNT'], q=3, labels=False, duplicates='drop')
group_modes = df.groupby('GIFT_COUNT_TIER')['PEP_STAR'].transform(lambda x: x.mode()[0] if not x.mode().empty else 0)
df['PEP_STAR'] = df['PEP_STAR'].fillna(group_modes).round().astype(int)
df.drop(columns=['GIFT_COUNT_TIER'], inplace=True)
print("--- PEP_STAR -- End ---")

print("--- FREQUENCY_STATUS_97NK --")
df['FREQUENCY_STATUS_97NK'] = df['FREQUENCY_STATUS_97NK'].apply(lambda x: np.nan if x < 1 else (4.0 if x > 4 else x))
df['FREQUENCY_STATUS_97NK_IS_MISSING'] = np.where(df['FREQUENCY_STATUS_97NK'].isna(), 1, 0)
df['FREQUENCY_STATUS_97NK'] = df['FREQUENCY_STATUS_97NK'].fillna(-1).round().astype(int)
print("--- FREQUENCY_STATUS_97NK -- End ---")

print("--- CHILDREN ---")
df['CHILDREN'] = df['CHILDREN'].apply(lambda x: np.nan if x < 0 else (4.0 if x > 4 else x))
df['CHILDREN_IS_MISSING'] = np.where(df['CHILDREN'].isna(), 1, 0)
df['CHILDREN'] = df['CHILDREN'].fillna(0).round().astype(int)
print("--- CHILDREN -- End ---")

print("--- INCOME_GROUP -- Start ---")
df['INCOME_GROUP'] = np.where(df['INCOME_GROUP'] > 7, 7, df['INCOME_GROUP'])
df['INCOME_GROUP'] = np.where((df['INCOME_GROUP'] < 1) | (df['INCOME_GROUP'].isna()), -1, df['INCOME_GROUP'])
df['INCOME_GROUP'] = df['INCOME_GROUP'].round().astype(int)
print("--- INCOME_GROUP -- End ---")

print("--- WEALTH_RATING -- Start ---")
df['WEALTH_RATING'] = np.where(df['WEALTH_RATING'] > 9, 9.0, df['WEALTH_RATING'])
df['WEALTH_RATING'] = np.where((df['WEALTH_RATING'] < 0) | (df['WEALTH_RATING'].isna()), -1.0, df['WEALTH_RATING'])
df['WEALTH_RATING'] = df['WEALTH_RATING'].round().astype(int)
print("--- WEALTH_RATING -- End ---")

print("--- RECENT_CARD_RESPONSE_COUNT ---")
df['RECENT_CARD_RESPONSE_COUNT_IS_MISSING'] = np.where(df['RECENT_CARD_RESPONSE_COUNT'].isna(), 1, 0)
df['RECENT_CARD_RESPONSE_COUNT'] = np.where(
    df['RECENT_CARD_RESPONSE_COUNT'] > 0, 
    df['RECENT_CARD_RESPONSE_COUNT'].round(), 
    df['RECENT_CARD_RESPONSE_COUNT']
)
df['RECENT_CARD_RESPONSE_COUNT'] = np.where(
    (df['RECENT_CARD_RESPONSE_COUNT'] < 0) | (df['RECENT_CARD_RESPONSE_COUNT'].isna()), 
    0, 
    df['RECENT_CARD_RESPONSE_COUNT']
).astype(int)
print("--- RECENT_CARD_RESPONSE_COUNT -- End ---")

print("--- RECENT_RESPONSE_COUNT ---")
df['RECENT_RESPONSE_COUNT_IS_MISSING'] = np.where(df['RECENT_RESPONSE_COUNT'].isna(), 1, 0)
df['RECENT_RESPONSE_COUNT'] = np.where(
    df['RECENT_RESPONSE_COUNT'] > 0, 
    df['RECENT_RESPONSE_COUNT'].round(), 
    df['RECENT_RESPONSE_COUNT']
)
df['RECENT_RESPONSE_COUNT'] = np.where(
    (df['RECENT_RESPONSE_COUNT'] < 0) | (df['RECENT_RESPONSE_COUNT'].isna()), 
    0, 
    df['RECENT_RESPONSE_COUNT']
).astype(int)
print("--- RECENT_RESPONSE_COUNT -- End ---")

print("--- CARD_PROM_12 ---")
df['CARD_PROM_12_IS_MISSING'] = np.where(df['CARD_PROM_12'].isna(), 1, 0)
df['CARD_PROM_12'] = np.where(df['CARD_PROM_12'].notna(), df['CARD_PROM_12'].round(), df['CARD_PROM_12'])
df['CARD_PROM_12'] = df['CARD_PROM_12'].fillna(0).astype(int)
print("--- CARD_PROM_12 -- End ---")

print("--- RECENT_STAR_STATUS -- Start ---")
df['RECENT_STAR_STATUS'] = np.where(df['RECENT_STAR_STATUS'].isna(), 1, 0)
df['RECENT_STAR_STATUS'] = np.where(df['RECENT_STAR_STATUS'] < 0, 0, df['RECENT_STAR_STATUS'])
df['RECENT_STAR_STATUS'] = np.where(df['RECENT_STAR_STATUS'] > 0, 1, df['RECENT_STAR_STATUS'])
df['RECENT_STAR_STATUS'] = df['RECENT_STAR_STATUS'].astype(int)
print("--- RECENT_STAR_STATUS -- End ---")

print("--- DONOR_AGE -- Start ---")
df['DONOR_AGE_IS_MISSING'] = np.where(df['DONOR_AGE'].isna(), 1, 0)
# Segment continuous columns into 3 groups (Low, Med, High tiers)
# This creates clean groups even if wealth rating is empty
df['INCOME_TIER'] = pd.qcut(df['MEDIAN_HOUSEHOLD_INCOME'], q=3, labels=['Low_Inc', 'Med_Inc', 'High_Inc'])
df['TENURE_TIER'] = pd.qcut(df['MONTHS_SINCE_FIRST_GIFT'], q=3, labels=['New_Donor', 'Mid_Donor', 'Loyal_Donor'])
robust_group_cols = ['URBANICITY', 'HOME_OWNER', 'INCOME_TIER', 'TENURE_TIER']
group_medians = df.groupby(robust_group_cols)['DONOR_AGE'].transform('median')
df['DONOR_AGE'] = df['DONOR_AGE'].fillna(group_medians)
overall_median = df['DONOR_AGE'].median() # fallback if any group is empty
df['DONOR_AGE'] = df['DONOR_AGE'].fillna(overall_median)
df['DONOR_AGE'] = df['DONOR_AGE'].round().astype(int)
df.drop(columns=['INCOME_TIER', 'TENURE_TIER'], inplace=True)
print("--- DONOR_AGE -- End ---")

# Cleaning remaining columns
discrete_cols = [
    'MONTHS_SINCE_LAST_GIFT', 'FILE_CARD_GIFT', 'MONTHS_SINCE_LAST_PROM_RESP',
    'NUMBER_PROM_12', 'LIFETIME_CARD_PROM', 'LIFETIME_GIFT_COUNT',
    'MONTHS_SINCE_FIRST_GIFT', 'LIFETIME_PROM'
]
for col in discrete_cols:
    df[col] = np.round(pd.to_numeric(df[col], errors='coerce'))
    df[col] = df[col].clip(lower=0).fillna(df[col].median()).astype(int)

proportions_n_percentages_cols = [
    'RECENT_CARD_RESPONSE_PROP', 'PCT_ATTRIBUTE1', 'PCT_ATTRIBUTE2', 
    'PCT_ATTRIBUTE3', 'PCT_ATTRIBUTE4', 'PCT_OWNER_OCCUPIED', 'RECENT_RESPONSE_PROP'
]
for col in proportions_n_percentages_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    # Check if the column is a 0-1 proportion or a 0-100 percentage
    if df[col].max() <= 1.0:
        df[col] = df[col].clip(lower=0.0, upper=1.0)
    else:
        df[col] = df[col].clip(lower=0.0, upper=100.0)
    df[col] = df[col].fillna(df[col].median())

monetary_cols = [
    'LIFETIME_MIN_GIFT_AMT', 'LAST_GIFT_AMT', 'LIFETIME_MAX_GIFT_AMT', 
    'RECENT_AVG_CARD_GIFT_AMT', 'RECENT_AVG_GIFT_AMT', 'LIFETIME_GIFT_AMOUNT', 
    'MEDIAN_HOUSEHOLD_INCOME', 'MEDIAN_HOME_VALUE', 'PER_CAPITA_INCOME'
]
for col in monetary_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].clip(lower=0.01) #expecting donations can't be zero.
    df[col] = df[col].fillna(df[col].median())

# print("--- SES from INCOME_GROUP ---")
# df['SES'] = df.groupby('INCOME_GROUP')['SES'].transform(
#     lambda x: x.fillna(x.mode()[0] if not x.mode().empty else df['SES'].median())
# )

print("--- URBANICITY from MEDIAN_HOME_VALUE ---")
df['URBANICITY'] = df['URBANICITY'].replace('?', np.nan)
# group by home value quintiles to find the urbanicity mode
home_value_groups = pd.qcut(df['MEDIAN_HOME_VALUE'], q=5, duplicates='drop')
global_mode = df['URBANICITY'].mode()[0]
df['URBANICITY'] = df.groupby(home_value_groups)['URBANICITY'].transform(
    lambda g: g.fillna(g.mode()[0] if not g.mode().empty else global_mode)
)

print("--- verifying ---\n")
missing_pct = df.isnull().mean() * 100
print(missing_pct[missing_pct > 0].sort_values(ascending=False))
print("--- DONE! ---")

--- DONOR_GENDER -- Start --- 
--- DONOR_GENDER -- End --- 
--- HOME_OWNER -- Start ---
--- HOME_OWNER -- End ---
--- RECENCY_STATUS_96NK -- Start ---
--- RECENCY_STATUS_96NK -- End ---
--- SES -- Start ---
--- SES -- End ---
--- URBANICITY -- Start ---
--- URBANICITY -- End ---
--- PEP_STAR ---
--- PEP_STAR -- End ---
--- FREQUENCY_STATUS_97NK --
--- FREQUENCY_STATUS_97NK -- End ---
--- CHILDREN ---
--- CHILDREN -- End ---
--- INCOME_GROUP -- Start ---
--- INCOME_GROUP -- End ---
--- WEALTH_RATING -- Start ---
--- WEALTH_RATING -- End ---
--- RECENT_CARD_RESPONSE_COUNT ---
--- RECENT_CARD_RESPONSE_COUNT -- End ---
--- RECENT_RESPONSE_COUNT ---
--- RECENT_RESPONSE_COUNT -- End ---
--- CARD_PROM_12 ---
--- CARD_PROM_12 -- End ---
--- RECENT_STAR_STATUS -- Start ---
--- RECENT_STAR_STATUS -- End ---
--- DONOR_AGE -- Start ---
--- DONOR_AGE -- End ---
--- URBANICITY from MEDIAN_HOME_VALUE ---
--- verifying ---

Series([], dtype: float64)
--- DONE! ---


In [159]:
df.to_csv('./data/scratch_data_prep.out.csv', index=False)